In [1]:
import pandas as pd
import numpy as np

In [7]:
df=pd.read_excel('final data.xlsx')

In [8]:
df.head()

,DATE,DATA,LOCATION
0,2020-09-17 00:00:00,A person dancing along a station platform move...,"Kingswood Station, NSW"
1,2020-05-23 00:00:00,A tram collided with another tram at low speed...,"Flinders St, Vic"
2,2021-11-06 00:00:00,The battery bank in an uncommissioned shunting...,"Hornsby, NSW"
3,2020-09-19 00:00:00,A freight train struck a pedestrian in the vic...,"Port Augusta, SA"
4,2020-08-14 00:00:00,An ambulance crossed onto the tram line in the...,"Thebarton, SA"


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 848 entries, 0 to 847
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   DATE      846 non-null    object
 1   DATA      846 non-null    object
 2   LOCATION  841 non-null    object
dtypes: object(3)
memory usage: 20.0+ KB


In [10]:
df.isna().sum()

DATE        2
DATA        2
LOCATION    7
dtype: int64

In [11]:
df.dropna(inplace=True)

In [12]:
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Assuming your DataFrame is named 'df' and the column containing the text is named 'text_column'

# Load the NLTK English stopwords
stop_words = set(stopwords.words('english'))

# Define a function to remove stop words from a given text
def remove_stop_words(text):
    tokens = word_tokenize(text)  # Tokenize the text into individual words
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]  # Filter out stop words
    return ' '.join(filtered_tokens)  # Join the filtered tokens back into a string

# Apply the remove_stop_words function to the text_column and store the results in a new column
df['triggering_factor'] = df['triggering_factor'].apply(remove_stop_words)

# Display the resulting DataFrame with stop words removed
print(df['triggering_factor'])


KeyError: 'triggering_factor'

In [30]:
import nltk
from nltk.corpus import stopwords  #stopwords
from nltk.stem import WordNetLemmatizer  
from sklearn.feature_extraction.text import TfidfVectorizer
stop_words=nltk.corpus.stopwords.words('english')

In [31]:
def clean_text(headline):
    le=WordNetLemmatizer()
    word_tokens=word_tokenize(headline)
    tokens=[le.lemmatize(w) for w in word_tokens if w not in stop_words and len(w)>3]
    cleaned_text=" ".join(tokens)
    return cleaned_text
df['triggering_factor']=df['triggering_factor'].apply(clean_text)

In [32]:
vect =TfidfVectorizer(stop_words=stop_words,max_features=1000)
vect_text=vect.fit_transform(df['triggering_factor'])

In [33]:
from sklearn.decomposition import LatentDirichletAllocation
lda_model=LatentDirichletAllocation(n_components=4,learning_method='online',random_state=42,max_iter=1) 
lda_top=lda_model.fit_transform(vect_text)

In [34]:
print("Document 0: ")
for i,topic in enumerate(lda_top[0]):
    print("Topic ",i,": ",topic*100,"%")

Document 0: 
Topic  0 :  7.400871617735939 %
Topic  1 :  7.367385466203463 %
Topic  2 :  77.81507042360207 %
Topic  3 :  7.416672492458535 %


In [35]:
vocab = vect.get_feature_names_out()
for i, comp in enumerate(lda_model.components_):
    vocab_comp = zip(vocab, comp)
    sorted_words = sorted(vocab_comp, key= lambda x:x[1], reverse=True)[:10]
    print("Topic "+str(i)+": ",end='\n')
    for t in sorted_words:
        print(t[0],end="\n")

Topic 0: 
tnatural
jumped
signal
india
temperature
mail
save
said
platform
telephone
Topic 1: 
firework
tnaturals
collided
overshot
detached
sabotage
signal
derailment
fast
save
Topic 2: 
tnatural
coach
derailed
fire
town
missing
disruption
express
collided
vehicle
Topic 3: 
natural
reason
speed
exact
known
snag
tnatural
india
level
boulder


In [36]:
import numpy as np
import pandas as pd
import re , nltk ,gensim,spacy
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from pprint import pprint
import pyLDAvis
import pyLDAvis.gensim_models
import matplotlib.pyplot as plt
%matplotlib inline

C:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\debug\cli\debugger_cli_common.py:19: DeprecationWarning: module 'sre_constants' is deprecated
  import sre_constants


In [37]:
data = df.triggering_factor

In [38]:
def sent_to_words(sentences):
    for sentence in sentences:
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))  # deacc=True removes punctuations
data_words = list(sent_to_words(data))
print(data_words[:1])

[['attack', 'godhra', 'station', 'gujarat', 'four', 'coach', 'fire']]


In [39]:
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']): #'NOUN', 'ADJ', 'VERB', 'ADV'
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append(" ".join([token.lemma_ if token.lemma_ not in ['-PRON-'] else '' for token in doc if token.pos_ in allowed_postags]))
    return texts_out

In [40]:
# Initialize spacy ‘en’ model, keeping only tagger component (for efficiency)
# Run in terminal: python -m spacy download en
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
# Do lemmatization keeping only Noun, Adj, Verb, Adverb
data_lemmatized = lemmatization(data_words, allowed_postags=['NOUN', 'VERB']) #select noun and verb
print(data_lemmatized[:2])

['attack godhra station coach fire', 'crash occur sabotage derail']


In [41]:
vectorizer = CountVectorizer(analyzer='word',min_df=3,stop_words='english',lowercase=True,token_pattern='[a-zA-Z0-9]{3,}')
data_vectorized = vectorizer.fit_transform(data_lemmatized)

In [27]:
# Build LDA Model
lda_model = LatentDirichletAllocation(n_components=4,               # Number of topics
                                      max_iter=10,               
# Max learning iterations
                                      learning_method='online',   
                                      random_state=100,          
# Random state
                                      batch_size=128,            
# n docs in each learning iter
                                      evaluate_every = -1,       
# compute perplexity every n iters, default: Don't
                                     )
lda_output = lda_model.fit_transform(data_vectorized)
#print(lda_model)  # Model attributes

In [30]:
# Create Document — Topic Matrix
lda_output = lda_model.transform(data_vectorized)
# column names
topicnames = ['Topic' + str(i) for i in range(lda_model.n_components)]
# index names
docnames = ['Doc' + str(i) for i in range(len(data))]
# Make the pandas dataframe
df_document_topic = pd.DataFrame(np.round(lda_output, 2), columns=topicnames, index=docnames)
# Get dominant topic for each document
dominant_topic = np.argmax(df_document_topic.values, axis=1)
df_document_topic['dominant_topic'] = dominant_topic
# Styling
def color_green(val):
    color = 'green' if val > .1 else 'black'
    return 'color: {col}'.format(col=color)
def make_bold(val):
    weight = 700 if val > .1 else 400
    return 'font-weight: {weight}'.format(weight=weight)
# Apply Style
df_document_topics = df_document_topic.head(15).style.applymap(color_green).applymap(make_bold)
df_document_topics

,Topic0,Topic1,Topic2,Topic3,dominant_topic
Doc0,0.750000,0.080000,0.080000,0.080000,0
Doc1,0.750000,0.080000,0.080000,0.080000,0
Doc2,0.080000,0.090000,0.090000,0.740000,3
Doc3,0.870000,0.040000,0.040000,0.040000,0
Doc4,0.620000,0.130000,0.130000,0.130000,0
Doc5,0.750000,0.080000,0.080000,0.080000,0
Doc6,0.050000,0.850000,0.050000,0.050000,1
Doc7,0.130000,0.620000,0.130000,0.130000,1
Doc8,0.620000,0.130000,0.130000,0.130000,0
Doc9,0.080000,0.080000,0.750000,0.080000,2


In [31]:
# Show top n keywords for each topic
def show_topics(vectorizer=vectorizer, lda_model=lda_model, n_words=20):
    keywords = np.array(vectorizer.get_feature_names_out())
    topic_keywords = []
    for topic_weights in lda_model.components_:
        top_keyword_locs = (-topic_weights).argsort()[:n_words]
        topic_keywords.append(keywords.take(top_keyword_locs))
    return topic_keywords
topic_keywords = show_topics(vectorizer=vectorizer, lda_model=lda_model, n_words=15)
# Topic - Keywords Dataframe
df_topic_keywords = pd.DataFrame(topic_keywords)
df_topic_keywords.columns = ['Word '+str(i) for i in range(df_topic_keywords.shape[1])]
df_topic_keywords.index = ['Topic '+str(i) for i in range(df_topic_keywords.shape[0])]
df_topic_keywords

,Word 0,Word 1,Word 2,Word 3,Word 4,Word 5,Word 6,Word 7,Word 8,Word 9,Word 10,Word 11,Word 12,Word 13,Word 14
Topic 0,derail,coach,tnatural,station,line,sabotage,fall,derailment,driver,reason,collide,railway,cause,level,passenger
Topic 1,cause,driver,derailment,signal,accident,passenger,railway,compartment,track,line,jump,sabotage,fall,station,tnatural
Topic 2,tnatural,track,collide,reason,cancel,passenger,fall,jump,railway,coach,accident,crossing,level,driver,derail
Topic 3,crossing,level,collide,passenger,driver,derail,signal,sabotage,jump,line,cancel,accident,compartment,tnatural,station


In [32]:
# Topic-Keyword Matrix
df_topic_keywords = pd.DataFrame(lda_model.components_)
# Assign Column and Index
df_topic_keywords.columns = vectorizer.get_feature_names_out()
df_topic_keywords.index = topicnames
# View
df_topic_keywords.head()

,accident,cancel,cause,coach,collide,compartment,crossing,derail,derailment,driver,...,level,line,passenger,railway,reason,sabotage,signal,station,tnatural,track
Topic0,0.368978,0.406825,0.471095,8.863488,0.513467,0.404344,0.390965,11.993883,0.810023,0.658652,...,0.435042,2.844850,0.407713,0.471225,0.553899,1.905141,0.386603,3.640235,4.543543,0.381394
Topic1,4.900591,0.386495,5.976030,0.377423,0.372035,2.828542,0.381640,0.393694,5.570065,5.663334,...,0.405228,1.939693,3.656210,2.829865,0.430043,1.199777,5.150705,1.162836,0.513592,1.958887
Topic2,0.618444,2.813952,0.407197,0.759305,3.481064,0.376369,0.529819,0.449900,0.439856,0.450194,...,0.483667,0.393297,2.769137,1.127373,3.375094,0.385826,0.393260,0.379780,13.867370,6.032384
Topic3,0.419629,0.445307,0.401558,0.378268,2.818623,0.416128,3.537558,0.808307,0.383193,1.248448,...,3.525435,0.458723,2.020301,0.398263,0.396175,0.515960,0.546326,0.402075,0.404280,0.386586


In [44]:
import gensim
import gensim.corpora as corpora
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from gensim.models.ldamodel import LdaModel

from pprint import pprint

import spacy

import pickle
import re
import pyLDAvis
import pyLDAvis.gensim

import matplotlib.pyplot as plt
import pandas as pd

In [58]:
#tweets = pd.read_csv('') #Change this with the name of your downloaded file
tweets = df.triggering_factor.values.tolist()

# Turn the list of string into a list of tokens
tweets = [t.split(' ') for t in tweets]

In [59]:
tweets

[['attack',
  'by',
  'a',
  'mob',
  'at',
  'Godhra',
  'station',
  'in',
  'Gujarat',
  'and',
  'four',
  'coaches',
  'were',
  'set',
  'on',
  'fire'],
 ['',
  'crash',
  'occurred',
  'when',
  'sabotage',
  'derailed',
  'the',
  'Shramjeevi',
  'express',
  'at',
  'Jaunpur'],
 ['Express',
  'collided',
  'with',
  'a',
  'passenger',
  'bus',
  'near',
  'the',
  'town',
  'of',
  'Kasganj'],
 ['derailed',
  'on',
  'a',
  'bridge',
  'between',
  'Gaya',
  'and',
  'Dehri-on-Sone',
  'stations|with',
  'two',
  'coaches',
  'falling',
  'into',
  'a',
  'river|terrorist',
  'sabotage',
  'was',
  'blamed'],
 ['three', 'coaches', 'caught', 'fire'],
 ['derailed', 'when', 'it', 'struck', 'a', 'boulder', 'on', 'the', 'line'],
 ['The',
  'accident',
  'was',
  'allegedly',
  'caused',
  'due',
  'to',
  'a',
  'fault',
  'in',
  'the',
  'telephone',
  'line|preventing',
  'proper',
  'signal',
  'warnings'],
 ['engine',
  'was',
  'halted',
  'immediately',
  'and',
  'derailm

In [60]:
id2word = Dictionary(tweets)
# Term Document Frequency
corpus = [id2word.doc2bow(text) for text in tweets]
print(corpus[:1])


[[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1)]]


In [61]:
[[(id2word[i], freq) for i, freq in doc] for doc in corpus[:1]]

[[('Godhra', 1),
  ('Gujarat', 1),
  ('a', 1),
  ('and', 1),
  ('at', 1),
  ('attack', 1),
  ('by', 1),
  ('coaches', 1),
  ('fire', 1),
  ('four', 1),
  ('in', 1),
  ('mob', 1),
  ('on', 1),
  ('set', 1),
  ('station', 1),
  ('were', 1)]]

In [62]:
lda_model = LdaModel(corpus=corpus,
                   id2word=id2word,
                   num_topics=10,
                   random_state=0,
                   chunksize=100,
                   alpha='auto',
                   per_word_topics=True)

pprint(lda_model.print_topics())
doc_lda = lda_model[corpus]

[(0,
  '0.060*"tnatural" + 0.052*"a" + 0.032*"the" + 0.028*"derailed" + 0.025*"and" '
  '+ 0.022*"at" + 0.020*"was" + 0.020*"in" + 0.017*"of" + 0.017*"by"'),
 (1,
  '0.033*"of" + 0.033*"the" + 0.024*"derailed" + 0.024*"tnatural" + '
  '0.023*"after" + 0.017*"line" + 0.016*"in" + 0.016*"to" + 0.015*"due" + '
  '0.009*"on"'),
 (2,
  '0.059*"tnaturals" + 0.040*"coaches" + 0.031*"cancelled" + 0.027*"were" + '
  '0.027*"10" + 0.027*"derailed" + 0.021*"the" + 0.014*"at" + 0.014*"affected" '
  '+ 0.014*"in"'),
 (3,
  '0.070*"the" + 0.044*"was" + 0.024*"a" + 0.021*"on" + 0.021*"bus" + '
  '0.021*"and" + 0.021*"driver" + 0.014*"tnatural" + 0.014*"coaches" + '
  '0.014*"engine"'),
 (4,
  '0.075*"the" + 0.033*"tnatural" + 0.020*"derailed" + 0.020*"is" + '
  '0.020*"cause" + 0.020*"not" + 0.020*"when" + 0.018*"and" + '
  '0.015*"stationary" + 0.014*"fire"'),
 (5,
  '0.033*"in" + 0.032*"track" + 0.032*"railway" + 0.032*"to" + 0.024*"on" + '
  '0.022*"the" + 0.022*"of" + 0.022*"due" + 0.013*"were" +

In [64]:
coherence_model_lda = CoherenceModel(model=lda_model, texts=tweets, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('nCoherence Score: ', coherence_lda)



nCoherence Score:  0.3733766618378551


In [65]:
#Creating Topic Distance Visualization 
pyLDAvis.enable_notebook()
p = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
p

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0     -0.079022 -0.115437       1        1  18.383563
3     -0.073908 -0.033436       2        1  14.994970
6     -0.052267  0.144985       3        1  13.013009
1      0.095473 -0.047825       4        1  11.487498
7     -0.062936  0.018820       5        1  10.900209
5      0.044516 -0.009540       6        1   7.407806
4     -0.015817 -0.009015       7        1   6.627110
9      0.096520  0.041301       8        1   6.415412
8     -0.033066  0.024532       9        1   6.284975
2      0.080507 -0.014384      10        1   4.485448, topic_info=         Term       Freq      Total Category  logprob  loglift
82  tnaturals   3.000000   3.000000  Default  30.0000  30.0000
24        the  42.000000  42.000000  Default  29.0000  29.0000
15       were   5.000000   5.000000  Default  28.0000  28.0000
7     coaches   7.000000   7.000000  Default  27.0000  27.0000
2           a  15.000000  15.000000  Default  26.0000  26.0000
..        ...        ...        ...      ...      ...      ...
10         in   0.502977   9.765947  Topic10  -4.2511   0.1382
24        the   0.754434  42.046045  Topic10  -3.8457  -0.9162
62         to   0.246984   7.942533  Topic10  -4.9623  -0.3663
77        for   0.203754   3.363973  Topic10  -5.1547   0.3004
2           a   0.198819  15.583381  Topic10  -5.1793  -1.2572

[479 rows x 6 columns], token_table=      Topic      Freq   Term
term                        
16        7  0.872749       
323      10  0.640035     10
93        4  0.731582     11
143       3  0.721523    141
367       3  0.721510      2
...     ...       ...    ...
34        6  0.182543   with
34        7  0.182543   with
253       6  0.800740  worst
352       8  0.809306    yet
149       3  0.721523     an

[508 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 4, 7, 2, 8, 6, 5, 10, 9, 3])